# Tugas Deep Learning: Scene Classification
## Custom CNN from Scratch vs Transfer Learning (MobileNetV2)
**Dataset:** Intel Image Classification (`puneet6060/intel-image-classification`)  
**6 Kelas:** `buildings`, `forest`, `glacier`, `mountain`, `sea`, `street`  
**Framework:** TensorFlow 2.15+ / Keras 3 | Python 3.10+  
**Target Hardware:** Google Colab Free GPU (NVIDIA Tesla T4)

---

### Deskripsi Proyek
Notebook ini menyajikan implementasi *end-to-end* perbandingan antara dua paradigma pemodelan *deep learning* untuk tugas klasifikasi citra pemandangan alam dan perkotaan (*Scene Classification*):
1. **Arsitektur 1 (Custom CNN from Scratch):** Model konvolusional hierarkis 4 blok dengan Batch Normalization, Max Pooling, Spatial Dropout, dan Global Average Pooling.
2. **Arsitektur 2 (Transfer Learning MobileNetV2):** Model terlatih (*pre-trained*) pada ImageNet dengan strategi dua tahap: *Feature Extraction* (bobot dasar dibekukan) dilanjutkan dengan *Fine-Tuning* pada 30 *layer* teratas menggunakan *learning rate* sangat kecil.

Semua pengujian dilakukan secara adil pada **set pengujian (*test set*) yang identik**, tanpa *data leakage*, dengan seed reproduktif global (42).


In [ ]:
# Verifikasi Lingkungan Komputasi & GPU Colab
import os
import random
import time
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

print(f"TensorFlow Version : {tf.__version__}")
print(f"Keras Version      : {tf.keras.__version__}")

# Cek ketersediaan akselerator GPU (T4 pada Colab Free)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        gpu_name = tf.test.gpu_device_name()
        print(f"[+] GPU Terdeteksi: {gpus} ({gpu_name})")
    except RuntimeError as e:
        print(f"[!] Error konfigurasi memori GPU: {e}")
else:
    print("[!] PERINGATAN: GPU tidak terdeteksi! Pastikan Runtime -> Change runtime type -> T4 GPU aktif.")

# Konfigurasi Direktori Kerja di Colab
OUTPUT_DIR = "/content/outputs"
CHECKPOINT_DIR = "/content/checkpoints"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"[+] Direktori artefak siap: {OUTPUT_DIR} & {CHECKPOINT_DIR}")

# Global Reproducibility Seed
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print(f"[+] Seed global berhasil disetel ke: {SEED}")


In [ ]:
# Instalasi library resmi pengunduh Kaggle
!pip -q install kagglehub


---
## 1. Unduh & Resolusi Struktur Dataset (Intel Image Classification)

Dataset Intel Image Classification memiliki kekhasan struktur direktori bertingkat ganda (*double-nested*):
```text
<dataset_root>/
├── seg_train/
│   └── seg_train/
│       ├── buildings/
│       ├── forest/
│       ├── glacier/
│       ├── mountain/
│       ├── sea/
│       └── street/
├── seg_test/
│   └── seg_test/
│       └── ...
└── seg_pred/
    └── seg_pred/ (diabaikan karena tidak berlabel)
```

Fungsi `resolve_dataset_dirs()` berikut secara rekursif menelusuri pohon direktori untuk menemukan path `seg_train` dan `seg_test` yang memuat subfolder kelas target, tanpa bergantung pada nomor versi atau kedalaman folder hardcoded.


In [ ]:
# Helper Downloader & Recursive Folder Resolver
EXPECTED_CLASSES = {"buildings", "forest", "glacier", "mountain", "sea", "street"}

def is_class_dir(directory: Path, expected: set = EXPECTED_CLASSES) -> bool:
    """Cek apakah direktori memuat subfolder kelas target."""
    if not directory.is_dir():
        return False
    subdirs = {d.name.lower() for d in directory.iterdir() if d.is_dir()}
    return len(expected.intersection(subdirs)) >= 4

def resolve_dataset_dirs(root: str) -> tuple[str, str]:
    """Menelusuri direktori root secara rekursif untuk mendeteksi folder seg_train dan seg_test yang benar,
    mengatasi permasalahan double-nested folder (seg_train/seg_train).
    """
    root_path = Path(root).resolve()
    if not root_path.exists():
        raise FileNotFoundError(f"Direktori root tidak ditemukan: {root_path}")
    
    train_candidates = []
    test_candidates = []
    
    for path in root_path.rglob("*"):
        if not path.is_dir():
            continue
        name_lower = path.name.lower()
        if "seg_train" in name_lower and is_class_dir(path):
            train_candidates.append(path)
        elif "seg_test" in name_lower and is_class_dir(path):
            test_candidates.append(path)
            
    # Fallback pencarian fleksibel
    if not train_candidates:
        for path in root_path.rglob("*"):
            if path.is_dir() and "train" in str(path).lower() and is_class_dir(path):
                train_candidates.append(path)
    if not test_candidates:
        for path in root_path.rglob("*"):
            if path.is_dir() and ("test" in str(path).lower() or "val" in str(path).lower()) and is_class_dir(path):
                test_candidates.append(path)
                
    if not train_candidates:
        raise FileNotFoundError(f"Folder seg_train valid tidak ditemukan di bawah: {root_path}")
    if not test_candidates:
        raise FileNotFoundError(f"Folder seg_test valid tidak ditemukan di bawah: {root_path}")
        
    # Ambil kandidat dengan path terdalam (double-nested)
    train_dir = max(train_candidates, key=lambda p: len(p.parts))
    test_dir = max(test_candidates, key=lambda p: len(p.parts))
    
    return str(train_dir), str(test_dir)

def download_via_kagglehub(handle: str = "puneet6060/intel-image-classification") -> str:
    import kagglehub
    print(f"[*] Mengunduh dataset '{handle}' menggunakan kagglehub...")
    path = kagglehub.dataset_download(handle)
    print(f"[+] Selesai diunduh ke: {path}")
    return str(path)

def download_via_kaggle_cli(handle: str = "puneet6060/intel-image-classification", output_dir: str = "/content/dataset") -> str:
    import subprocess
    os.makedirs(output_dir, exist_ok=True)
    cmd = f"kaggle datasets download -d {handle} --unzip -p {output_dir}"
    print(f"[*] Menjalankan CLI: {cmd}")
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError(f"Gagal mengunduh via Kaggle CLI:\n{res.stderr}")
    return output_dir

def extract_from_drive_zip(zip_path: str, output_dir: str = "/content/dataset") -> str:
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f"Berkas ZIP tidak ditemukan: {zip_path}")
    os.makedirs(output_dir, exist_ok=True)
    print(f"[*] Mengekstrak {zip_path} ke {output_dir}...")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(output_dir)
    return output_dir


In [ ]:
# Eksekusi Unduh Dataset: Pilih Salah Satu Metode
KAGGLE_DATASET = "puneet6060/intel-image-classification"

# ==============================================================================
# OPSI A (REKOMENDASI): Kagglehub (Tanpa token manual jika dataset publik)
# ==============================================================================
raw_dataset_path = download_via_kagglehub(KAGGLE_DATASET)

# ==============================================================================
# OPSI B (ALTERNATIF): Kaggle CLI via Colab Secrets (KAGGLE_USERNAME & KAGGLE_KEY)
# ==============================================================================
# from google.colab import userdata
# os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
# os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
# raw_dataset_path = download_via_kaggle_cli(KAGGLE_DATASET, output_dir="/content/dataset")

# ==============================================================================
# OPSI C (ALTERNATIF): Google Drive ZIP
# ==============================================================================
# from google.colab import drive
# drive.mount('/content/drive')
# raw_dataset_path = extract_from_drive_zip('/content/drive/MyDrive/intel-image-classification.zip', output_dir="/content/dataset")

# Resolusi path seg_train & seg_test rekursif
train_dir, test_dir = resolve_dataset_dirs(raw_dataset_path)

print("-" * 60)
print(f"[+] Direktori Train Terverifikasi : {train_dir}")
print(f"[+] Direktori Test Terverifikasi  : {test_dir}")
print("-" * 60)


---
## 2. Exploratory Data Analysis (EDA) Ringkas

Tahap ini memverifikasi:
- Distribusi jumlah citra per kelas pada dataset *train* dan *test*.
- Keseimbangan (*balance*) sampel antar kelas.
- Tampilan 9 sampel citra pelatihan acak beserta labelnya.


In [ ]:
# Perhitungan Distribusi Citra dan Visualisasi 9 Sampel
train_path_obj = Path(train_dir)
test_path_obj = Path(test_dir)

train_counts = {p.name: len(list(p.glob("*.*"))) for p in train_path_obj.iterdir() if p.is_dir()}
test_counts = {p.name: len(list(p.glob("*.*"))) for p in test_path_obj.iterdir() if p.is_dir()}

classes = sorted(list(train_counts.keys()))
df_dist = pd.DataFrame({
    "Class": classes,
    "Train Count": [train_counts.get(c, 0) for c in classes],
    "Test Count": [test_counts.get(c, 0) for c in classes]
})

print("Distribusi Sampel per Kelas:")
print(df_dist.to_string(index=False))
print(f"Total Train Images: {sum(df_dist['Train Count'])}")
print(f"Total Test Images : {sum(df_dist['Test Count'])}")

# Grafik Batang Distribusi
fig, ax = plt.subplots(figsize=(10, 4.5))
x = np.arange(len(classes))
width = 0.35
ax.bar(x - width/2, df_dist["Train Count"], width, label="Train Set", color="#2b5c8f")
ax.bar(x + width/2, df_dist["Test Count"], width, label="Test Set", color="#e76f51")
ax.set_title("Distribusi Jumlah Citra per Kelas (Intel Image Classification)", fontsize=12, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(classes, rotation=20)
ax.set_ylabel("Jumlah Citra")
ax.legend()
ax.grid(axis='y', linestyle=':', alpha=0.6)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/eda_class_distribution.png", dpi=300)
plt.show()

# Visualisasi 9 Sampel Citra Pelatihan Acak (3x3 Grid)
fig, axes = plt.subplots(3, 3, figsize=(9, 9))
axes = axes.flatten()

all_train_images = []
for c in classes:
    for img_p in (train_path_obj / c).glob("*.*"):
        all_train_images.append((img_p, c))

random.seed(SEED)
sample_picks = random.sample(all_train_images, 9)

for idx, (img_path, class_lbl) in enumerate(sample_picks):
    img = Image.open(img_path)
    axes[idx].imshow(img)
    axes[idx].set_title(f"Label: {class_lbl}\nSize: {img.size}", fontsize=10)
    axes[idx].axis("off")

plt.suptitle("Sampel Citra Latih (Intel Image Classification)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/eda_sample_images.png", dpi=300)
plt.show()


---
## 3. Data Pipeline, Preprocessing, & Augmentasi

### Keputusan Desain:
- **Input Size:** 150x150 piksel (sesuai resolusi asli dataset, efisien pada GPU T4).
- **Batch Size:** 32.
- **Validation Split:** 20% diambil murni dari data latih (`seg_train`), bukan dari `seg_test`.
- **Test Set:** Terisolasi sepenuhnya dari fase pelatihan dan validasi (mencegah *data leakage*).
- **Augmentasi Data:** `RandomFlip('horizontal')`, `RandomRotation(0.1)`, dan `RandomZoom(0.1)` untuk regularisasi spasial.
- **Normalisasi:**
  - *Custom CNN*: `Rescaling(1./255)` ke rentang $[0.0, 1.0]$.
  - *MobileNetV2*: `mobilenet_v2.preprocess_input` ke rentang $[-1.0, 1.0]$.
- Optimasi I/O: `.cache()` dan `.prefetch(tf.data.AUTOTUNE)`.


In [ ]:
# Pembangunan tf.data Dataset Pipeline
IMG_SIZE = (150, 150)
BATCH_SIZE = 32
VAL_SPLIT = 0.2

print("[*] Memuat dataset menggunakan tf.keras.utils.image_dataset_from_directory...")

# 1. Dataset Mentah (Raw uint8)
raw_train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=VAL_SPLIT,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=True
)

raw_val_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=VAL_SPLIT,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

raw_test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

class_names = raw_train_ds.class_names
NUM_CLASSES = len(class_names)
print(f"[+] Berhasil memuat {NUM_CLASSES} kelas: {class_names}")

# 2. Data Augmentation Layer (Hanya untuk Training)
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal", name="aug_flip"),
    tf.keras.layers.RandomRotation(0.1, name="aug_rotation"),
    tf.keras.layers.RandomZoom(0.1, name="aug_zoom")
], name="data_augmentation")

# 3. Pipeline untuk Custom CNN (Rescaling 1/255)
rescale_layer = tf.keras.layers.Rescaling(1.0 / 255.0, name="rescaling_cnn")

train_ds_cnn = raw_train_ds.map(
    lambda x, y: (rescale_layer(data_augmentation(x, training=True)), y),
    num_parallel_calls=tf.data.AUTOTUNE
).cache().prefetch(buffer_size=tf.data.AUTOTUNE)

val_ds_cnn = raw_val_ds.map(
    lambda x, y: (rescale_layer(x), y),
    num_parallel_calls=tf.data.AUTOTUNE
).cache().prefetch(buffer_size=tf.data.AUTOTUNE)

test_ds_cnn = raw_test_ds.map(
    lambda x, y: (rescale_layer(x), y),
    num_parallel_calls=tf.data.AUTOTUNE
).cache().prefetch(buffer_size=tf.data.AUTOTUNE)

# 4. Pipeline untuk MobileNetV2 (preprocess_input [-1, 1])
mobilenet_prep = tf.keras.applications.mobilenet_v2.preprocess_input

train_ds_mobilenet = raw_train_ds.map(
    lambda x, y: (mobilenet_prep(tf.cast(data_augmentation(x, training=True), tf.float32)), y),
    num_parallel_calls=tf.data.AUTOTUNE
).cache().prefetch(buffer_size=tf.data.AUTOTUNE)

val_ds_mobilenet = raw_val_ds.map(
    lambda x, y: (mobilenet_prep(tf.cast(x, tf.float32)), y),
    num_parallel_calls=tf.data.AUTOTUNE
).cache().prefetch(buffer_size=tf.data.AUTOTUNE)

test_ds_mobilenet = raw_test_ds.map(
    lambda x, y: (mobilenet_prep(tf.cast(x, tf.float32)), y),
    num_parallel_calls=tf.data.AUTOTUNE
).cache().prefetch(buffer_size=tf.data.AUTOTUNE)

print("[+] Pipeline tf.data untuk Custom CNN & MobileNetV2 siap dengan cache & prefetch.")


---
## 4. Arsitektur 1: Custom CNN from Scratch

### Spesifikasi Arsitektur:
- **Input:** Tensor berdimensi `(150, 150, 3)`.
- **4 Blok Konvolusional Bertingkat:**
  - *Blok 1:* Conv2D(32, 3x3) + BatchNorm + Conv2D(32, 3x3) + BatchNorm + MaxPool(2x2) + Dropout(0.2)
  - *Blok 2:* Conv2D(64, 3x3) + BatchNorm + Conv2D(64, 3x3) + BatchNorm + MaxPool(2x2) + Dropout(0.25)
  - *Blok 3:* Conv2D(128, 3x3) + BatchNorm + Conv2D(128, 3x3) + BatchNorm + MaxPool(2x2) + Dropout(0.3)
  - *Blok 4:* Conv2D(128, 3x3) + BatchNorm + MaxPool(2x2) + Dropout(0.3)
- **Top Classifier:**
  - GlobalAveragePooling2D() untuk mereduksi dimensionalitas spasial tanpa ledakan parameter.
  - Dense(128, activation='relu') + BatchNorm + Dropout(0.4)
  - Dense(6, activation='softmax', name='predictions')
- **Optimasi:** Adam (learning rate = $1 	imes 10^{-3}$), Sparse Categorical Crossentropy.


In [ ]:
# Definisi & Pembangunan Model Custom CNN
def build_custom_cnn(input_shape=(150, 150, 3), num_classes=6) -> tf.keras.Model:
    inputs = tf.keras.Input(shape=input_shape, name="input_tensor")
    
    # Blok 1
    x = tf.keras.layers.Conv2D(32, (3, 3), padding="same", activation="relu", name="conv1_1")(inputs)
    x = tf.keras.layers.BatchNormalization(name="bn1_1")(x)
    x = tf.keras.layers.Conv2D(32, (3, 3), padding="same", activation="relu", name="conv1_2")(x)
    x = tf.keras.layers.BatchNormalization(name="bn1_2")(x)
    x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2), name="pool1")(x)
    x = tf.keras.layers.Dropout(0.2, name="drop1")(x)
    
    # Blok 2
    x = tf.keras.layers.Conv2D(64, (3, 3), padding="same", activation="relu", name="conv2_1")(x)
    x = tf.keras.layers.BatchNormalization(name="bn2_1")(x)
    x = tf.keras.layers.Conv2D(64, (3, 3), padding="same", activation="relu", name="conv2_2")(x)
    x = tf.keras.layers.BatchNormalization(name="bn2_2")(x)
    x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2), name="pool2")(x)
    x = tf.keras.layers.Dropout(0.25, name="drop2")(x)
    
    # Blok 3
    x = tf.keras.layers.Conv2D(128, (3, 3), padding="same", activation="relu", name="conv3_1")(x)
    x = tf.keras.layers.BatchNormalization(name="bn3_1")(x)
    x = tf.keras.layers.Conv2D(128, (3, 3), padding="same", activation="relu", name="conv3_2")(x)
    x = tf.keras.layers.BatchNormalization(name="bn3_2")(x)
    x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2), name="pool3")(x)
    x = tf.keras.layers.Dropout(0.3, name="drop3")(x)
    
    # Blok 4
    x = tf.keras.layers.Conv2D(128, (3, 3), padding="same", activation="relu", name="conv4_1")(x)
    x = tf.keras.layers.BatchNormalization(name="bn4_1")(x)
    x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2), name="pool4")(x)
    x = tf.keras.layers.Dropout(0.3, name="drop4")(x)
    
    # Classification Head
    x = tf.keras.layers.GlobalAveragePooling2D(name="gap")(x)
    x = tf.keras.layers.Dense(128, activation="relu", name="dense1")(x)
    x = tf.keras.layers.BatchNormalization(name="bn_dense")(x)
    x = tf.keras.layers.Dropout(0.4, name="drop_dense")(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax", name="predictions")(x)
    
    return tf.keras.Model(inputs=inputs, outputs=outputs, name="CustomCNN")

model_cnn = build_custom_cnn(input_shape=(150, 150, 3), num_classes=NUM_CLASSES)
model_cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")]
)

model_cnn.summary()
trainable_cnn = sum(tf.keras.backend.count_params(w) for w in model_cnn.trainable_weights)
total_cnn = sum(tf.keras.backend.count_params(w) for w in model_cnn.weights)
print(f"[+] Custom CNN: Trainable Params = {trainable_cnn:,} | Total Params = {total_cnn:,}")


In [ ]:
# Pelatihan Custom CNN dengan Callbacks
EPOCHS = 30
PATIENCE = 5

cnn_checkpoint_path = f"{CHECKPOINT_DIR}/custom_cnn_best.keras"
callbacks_cnn = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=cnn_checkpoint_path,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

print("[*] Memulai pelatihan Custom CNN...")
start_time_cnn = time.perf_counter()

history_cnn = model_cnn.fit(
    train_ds_cnn,
    validation_data=val_ds_cnn,
    epochs=EPOCHS,
    callbacks=callbacks_cnn
)

time_cnn_sec = time.perf_counter() - start_time_cnn
print(f"[+] Selesai melatih Custom CNN dalam {time_cnn_sec:.2f} detik.")

# Simpan History ke CSV
df_history_cnn = pd.DataFrame(history_cnn.history)
df_history_cnn.index.name = "epoch"
df_history_cnn.reset_index(inplace=True)
df_history_cnn["epoch"] += 1
df_history_cnn.to_csv(f"{OUTPUT_DIR}/custom_cnn_history.csv", index=False)

# Visualisasi Kurva Pembelajaran (Loss & Accuracy)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
epochs_range_cnn = range(1, len(df_history_cnn) + 1)

# Loss
axes[0].plot(epochs_range_cnn, df_history_cnn["loss"], label="Train Loss", color="#1f77b4", linewidth=2)
axes[0].plot(epochs_range_cnn, df_history_cnn["val_loss"], label="Val Loss", color="#ff7f0e", linestyle="--", linewidth=2)
axes[0].set_title("Custom CNN - Loss", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, linestyle=":", alpha=0.6)

# Accuracy
acc_col = "accuracy" if "accuracy" in df_history_cnn else "sparse_categorical_accuracy"
val_acc_col = "val_accuracy" if "val_accuracy" in df_history_cnn else "val_sparse_categorical_accuracy"

axes[1].plot(epochs_range_cnn, df_history_cnn[acc_col], label="Train Accuracy", color="#2ca02c", linewidth=2)
axes[1].plot(epochs_range_cnn, df_history_cnn[val_acc_col], label="Val Accuracy", color="#d62728", linestyle="--", linewidth=2)
axes[1].set_title("Custom CNN - Accuracy", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/custom_cnn_learning_curves.png", dpi=300)
plt.show()


---
## 5. Arsitektur 2: Transfer Learning MobileNetV2 (ImageNet)

### Strategi Pelatihan Dua Tahap (*Two-Stage Training*):
1. **Tahap 1: Feature Extraction**
   - Backbone MobileNetV2 (ImageNet) dibekukan total (`base_model.trainable = False`).
   - Hanya kepala klasifikasi (*GlobalAveragePooling2D + Dropout(0.3) + Dense(6, softmax)*) yang dilatih.
   - *Learning rate:* $1 	imes 10^{-3}$ (Adam).
2. **Tahap 2: Fine-Tuning Terarah (30 Layer Terakhir)**
   - Unfreeze 30 layer teratas dari MobileNetV2.
   - Tetap pertahankan layer `BatchNormalization` dalam mode inferensi (`training=False`) agar estimasi statistik ImageNet tidak terdistorsi oleh ukuran batch kecil.
   - Dilatih ulang dengan *learning rate* konservatif ($1 	imes 10^{-5}$) guna memitigasi *catastrophic forgetting*.


In [ ]:
# Pembangunan & Pelatihan MobileNetV2 Tahap 1 (Feature Extraction)
base_mobilenet = tf.keras.applications.MobileNetV2(
    input_shape=(150, 150, 3),
    include_top=False,
    weights="imagenet"
)
base_mobilenet.trainable = False

inputs_mb = tf.keras.Input(shape=(150, 150, 3), name="mobilenet_input")
# training=False penting agar BatchNormalization tetap dalam mode inferensi
x_mb = base_mobilenet(inputs_mb, training=False)
x_mb = tf.keras.layers.GlobalAveragePooling2D(name="global_pool")(x_mb)
x_mb = tf.keras.layers.Dropout(0.3, name="top_dropout")(x_mb)
outputs_mb = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", name="predictions")(x_mb)

model_mobilenet = tf.keras.Model(inputs=inputs_mb, outputs=outputs_mb, name="MobileNetV2_TransferLearning")

model_mobilenet.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")]
)

model_mobilenet.summary()

# Training Tahap 1
mb_checkpoint_stage1 = f"{CHECKPOINT_DIR}/mobilenetv2_stage1.keras"
callbacks_mb_stage1 = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(filepath=mb_checkpoint_stage1, monitor="val_loss", save_best_only=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1)
]

print("[*] Memulai Pelatihan MobileNetV2 Tahap 1 (Feature Extraction)...")
start_time_mb = time.perf_counter()

history_mb_stage1 = model_mobilenet.fit(
    train_ds_mobilenet,
    validation_data=val_ds_mobilenet,
    epochs=15,
    callbacks=callbacks_mb_stage1
)


In [ ]:
# Pelatihan MobileNetV2 Tahap 2 (Fine-Tuning 30 Layer Terakhir)
base_mobilenet.trainable = True

# Bekukan semua layer kecuali 30 layer terakhir
fine_tune_at = len(base_mobilenet.layers) - 30
for layer in base_mobilenet.layers[:fine_tune_at]:
    layer.trainable = False

print(f"[+] Total layer di base model: {len(base_mobilenet.layers)}")
print(f"[+] Membuka bobot layer {fine_tune_at} ke atas untuk Fine-Tuning ({len(base_mobilenet.layers) - fine_tune_at} layer).")

# Re-compile dengan learning rate yang jauh lebih kecil
model_mobilenet.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")]
)

mb_checkpoint_best = f"{CHECKPOINT_DIR}/mobilenetv2_best.keras"
callbacks_mb_stage2 = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(filepath=mb_checkpoint_best, monitor="val_loss", save_best_only=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7, verbose=1)
]

initial_epochs = len(history_mb_stage1.history["loss"])
fine_tune_epochs = 15
total_epochs_mb = initial_epochs + fine_tune_epochs

print(f"[*] Melanjutkan ke Tahap 2: Fine-Tuning hingga epoch {total_epochs_mb}...")
history_mb_stage2 = model_mobilenet.fit(
    train_ds_mobilenet,
    validation_data=val_ds_mobilenet,
    epochs=total_epochs_mb,
    initial_epoch=initial_epochs,
    callbacks=callbacks_mb_stage2
)

time_mb_sec = time.perf_counter() - start_time_mb
print(f"[+] Total waktu pelatihan MobileNetV2 (Tahap 1 + 2): {time_mb_sec:.2f} detik.")

# Gabungkan Riwayat Training Tahap 1 dan Tahap 2
combined_history_mb = {}
for k in history_mb_stage1.history.keys():
    combined_history_mb[k] = history_mb_stage1.history[k] + history_mb_stage2.history[k]

df_history_mb = pd.DataFrame(combined_history_mb)
df_history_mb.index.name = "epoch"
df_history_mb.reset_index(inplace=True)
df_history_mb["epoch"] += 1
df_history_mb.to_csv(f"{OUTPUT_DIR}/mobilenetv2_history.csv", index=False)

# Visualisasi Kurva Pembelajaran Gabungan MobileNetV2
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
epochs_range_mb = range(1, len(df_history_mb) + 1)

# Loss
axes[0].plot(epochs_range_mb, df_history_mb["loss"], label="Train Loss", color="#1f77b4", linewidth=2)
axes[0].plot(epochs_range_mb, df_history_mb["val_loss"], label="Val Loss", color="#ff7f0e", linestyle="--", linewidth=2)
axes[0].axvline(x=initial_epochs, color="gray", linestyle=":", label="Mulai Fine-Tuning")
axes[0].set_title("MobileNetV2 - Loss (Feature Extraction + Fine-Tuning)", fontsize=11, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, linestyle=":", alpha=0.6)

# Accuracy
mb_acc_col = "accuracy" if "accuracy" in df_history_mb else "sparse_categorical_accuracy"
mb_val_acc_col = "val_accuracy" if "val_accuracy" in df_history_mb else "val_sparse_categorical_accuracy"

axes[1].plot(epochs_range_mb, df_history_mb[mb_acc_col], label="Train Accuracy", color="#2ca02c", linewidth=2)
axes[1].plot(epochs_range_mb, df_history_mb[mb_val_acc_col], label="Val Accuracy", color="#d62728", linestyle="--", linewidth=2)
axes[1].axvline(x=initial_epochs, color="gray", linestyle=":", label="Mulai Fine-Tuning")
axes[1].set_title("MobileNetV2 - Accuracy", fontsize=11, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/mobilenetv2_learning_curves.png", dpi=300)
plt.show()


---
## 6. Evaluasi Komparatif & Pengujian Independen

Kedua model dievaluasi secara ketat pada **test set yang SAMA**, dengan metrik:
1. **Test Loss & Test Accuracy**
2. **Classification Report:** Precision, Recall, F1-Score per kelas, serta rata-rata makro (*Macro Average*).
3. **Confusion Matrix Heatmap:** Mendiagnosis kelas-kelas yang saling membingungkan (misal: *glacier* vs *mountain*).
4. **Efisiensi Komputasi:** Jumlah parameter yang dapat dilatih (*trainable params*) dan durasi waktu pelatihan.


In [ ]:
# Evaluasi Menyeluruh Kedua Model pada Test Set
def run_evaluation(model, test_ds, model_name: str, training_time: float) -> dict:
    print(f"[*] Mengevaluasi {model_name}...")
    loss, acc = model.evaluate(test_ds, verbose=0)
    
    y_true_all = []
    y_pred_all = []
    
    for images, labels in test_ds:
        preds = model.predict(images, verbose=0)
        y_pred_all.append(np.argmax(preds, axis=1))
        y_true_all.append(labels.numpy())
        
    y_true = np.concatenate(y_true_all, axis=0)
    y_pred = np.concatenate(y_pred_all, axis=0)
    
    # Classification Report
    rep_str = classification_report(y_true, y_pred, target_names=class_names, digits=4, zero_division=0)
    rep_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0)
    
    rep_file = f"{OUTPUT_DIR}/{model_name.lower()}_classification_report.txt"
    with open(rep_file, "w", encoding="utf-8") as f:
        f.write(f"Classification Report - {model_name}\n")
        f.write("=" * 60 + "\n")
        f.write(rep_str)
        f.write("\n")
    print(f"[+] Laporan klasifikasi disimpan ke: {rep_file}")
    print(rep_str)
    
    # Confusion Matrix Heatmap
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(7.5, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Confusion Matrix - {model_name}", fontsize=12, fontweight="bold", pad=10)
    plt.xlabel("Predicted Class", fontsize=10)
    plt.ylabel("True Class", fontsize=10)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    cm_file = f"{OUTPUT_DIR}/{model_name.lower()}_confusion_matrix.png"
    plt.savefig(cm_file, dpi=300)
    plt.show()
    
    trainable_p = sum(tf.keras.backend.count_params(w) for w in model.trainable_weights)
    total_p = sum(tf.keras.backend.count_params(w) for w in model.weights)
    
    return {
        "Model": model_name,
        "Accuracy": float(acc),
        "Loss": float(loss),
        "Macro Precision": float(rep_dict["macro avg"]["precision"]),
        "Macro Recall": float(rep_dict["macro avg"]["recall"]),
        "Macro F1": float(rep_dict["macro avg"]["f1-score"]),
        "Training Time (s)": round(training_time, 2),
        "Trainable Params": int(trainable_p),
        "Total Params": int(total_p)
    }

# Evaluasi Custom CNN
metrics_cnn = run_evaluation(model_cnn, test_ds_cnn, "CustomCNN", time_cnn_sec)

# Evaluasi MobileNetV2
metrics_mobilenet = run_evaluation(model_mobilenet, test_ds_mobilenet, "MobileNetV2", time_mb_sec)


In [ ]:
# Tabel Perbandingan & Visualisasi Grafik Batang Benchmark
results_list = [metrics_cnn, metrics_mobilenet]
df_benchmark = pd.DataFrame(results_list)

# Simpan Tabel Benchmark ke CSV
benchmark_csv_path = f"{OUTPUT_DIR}/benchmark_results.csv"
df_benchmark.to_csv(benchmark_csv_path, index=False)
print(f"[+] Hasil perbandingan disimpan ke: {benchmark_csv_path}\n")

print("=" * 90)
print("TABEL PERBANDINGAN PERFORMA MODEL")
print("=" * 90)
display(df_benchmark)

# Grafik Batang Perbandingan Metrik
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes = axes.flatten()

bar_specs = [
    ("Accuracy", "Test Accuracy", "Akurasi", "#2b5c8f"),
    ("Loss", "Test Loss", "Loss", "#c23b22"),
    ("Macro F1", "Macro F1-Score", "F1-Score", "#2a9d8f"),
    ("Training Time (s)", "Waktu Pelatihan (detik)", "Detik", "#e76f51")
]

models = df_benchmark["Model"].tolist()

for idx, (col, title, ylabel, color) in enumerate(bar_specs):
    ax = axes[idx]
    vals = df_benchmark[col].tolist()
    bars = ax.bar(models, vals, color=color, width=0.45, edgecolor="black", alpha=0.85)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_ylabel(ylabel)
    ax.grid(axis='y', linestyle=':', alpha=0.6)
    
    for bar in bars:
        h = bar.get_height()
        lbl = f"{h:.4f}" if col != "Training Time (s)" else f"{h:.1f}s"
        ax.annotate(
            lbl,
            xy=(bar.get_x() + bar.get_width() / 2, h),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontweight="bold",
            fontsize=9
        )

plt.suptitle("Perbandingan Komprehensif: Custom CNN vs MobileNetV2", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/model_comparison_charts.png", dpi=300)
plt.show()


---
## 7. Model Interpretability dengan Grad-CAM (Gradient-weighted Class Activation Mapping)

Grad-CAM memvisualisasikan daerah spasial pada citra yang menjadi fokus perhatian model saat membuat keputusan prediksi kelas.
Di sini kita menguji 4 sampel citra uji pada **model dengan akurasi tertinggi** untuk memverifikasi apakah model mengidentifikasi fitur visual semantik yang tepat (misal: puncak bersalju untuk *mountain/glacier*, tekstur air laut untuk *sea*, daun/pohon untuk *forest*).


In [ ]:
# Implementasi & Visualisasi Grad-CAM
best_model_name = "MobileNetV2" if metrics_mobilenet["Accuracy"] >= metrics_cnn["Accuracy"] else "CustomCNN"
best_model = model_mobilenet if best_model_name == "MobileNetV2" else model_cnn
print(f"[+] Model terbaik terpilih untuk Grad-CAM: {best_model_name}")

def make_gradcam_heatmap(img_array, model, pred_index=None):
    """Menghitung activation map Grad-CAM untuk Custom CNN atau MobileNetV2."""
    if "MobileNetV2" in model.name:
        # Arsitektur nested: base_model di dalam outer model
        base_layer = model.get_layer("mobilenetv2_1.00_150")
        target_conv = base_layer.get_layer("out_relu") # Layer aktivasi terakhir MobileNetV2
        
        feature_model = tf.keras.Model([base_layer.inputs], [target_conv.output, base_layer.output])
        head_input = tf.keras.Input(shape=base_layer.output.shape[1:])
        x = head_input
        for layer in model.layers[2:]:
            x = layer(x)
        head_model = tf.keras.Model(head_input, x)
        
        with tf.GradientTape() as tape:
            conv_out, base_out = feature_model(img_array)
            tape.watch(conv_out)
            preds = head_model(base_out)
            if pred_index is None:
                pred_index = tf.argmax(preds[0])
            class_channel = preds[:, pred_index]
        grads = tape.gradient(class_channel, conv_out)
    else:
        # Custom CNN flat architecture
        target_layer = model.get_layer("conv4_1")
        grad_model = tf.keras.Model([model.inputs], [target_layer.output, model.output])
        with tf.GradientTape() as tape:
            conv_out, preds = grad_model(img_array)
            if pred_index is None:
                pred_index = tf.argmax(preds[0])
            class_channel = preds[:, pred_index]
        grads = tape.gradient(class_channel, conv_out)
        
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_out = conv_out[0]
    heatmap = conv_out @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0.0)
    max_val = tf.math.reduce_max(heatmap)
    if max_val > 0:
        heatmap /= max_val
    return heatmap.numpy()

def overlay_gradcam(orig_img_np, heatmap, alpha=0.4):
    """Superimpose heatmap Grad-CAM pada gambar asli."""
    heatmap_img = Image.fromarray(np.uint8(255 * heatmap))
    heatmap_resized = heatmap_img.resize((orig_img_np.shape[1], orig_img_np.shape[0]), resample=Image.BILINEAR)
    heatmap_resized = np.array(heatmap_resized) / 255.0
    
    cmap = plt.colormaps.get_cmap("jet")
    jet_colors = cmap(heatmap_resized)[:, :, :3]
    jet_heatmap = np.uint8(255 * jet_colors)
    
    superimposed = np.uint8(orig_img_np * (1.0 - alpha) + jet_heatmap * alpha)
    return superimposed

# Ambil 4 sampel gambar dari 4 kelas berbeda dari test_dir
sample_classes = ["forest", "glacier", "mountain", "sea"]
gradcam_samples = []

for c in sample_classes:
    cls_folder = Path(test_dir) / c
    files = list(cls_folder.glob("*.*"))
    if files:
        gradcam_samples.append((files[0], c))

# Visualisasi 4 Sampel: Citra Asli | Heatmap | Grad-CAM Overlay
fig, axes = plt.subplots(len(gradcam_samples), 3, figsize=(11, 3.2 * len(gradcam_samples)))

for idx, (img_p, true_cls) in enumerate(gradcam_samples):
    pil_img = Image.open(img_p).resize((150, 150))
    orig_np = np.array(pil_img)
    
    # Preprocessing sesuai model
    if best_model_name == "MobileNetV2":
        input_tensor = tf.keras.applications.mobilenet_v2.preprocess_input(np.expand_dims(orig_np.astype(np.float32), axis=0))
    else:
        input_tensor = np.expand_dims(orig_np.astype(np.float32) / 255.0, axis=0)
        
    preds = best_model.predict(input_tensor, verbose=0)
    pred_idx = np.argmax(preds[0])
    pred_cls = class_names[pred_idx]
    confidence = preds[0][pred_idx] * 100
    
    heatmap = make_gradcam_heatmap(input_tensor, best_model, pred_index=pred_idx)
    overlay = overlay_gradcam(orig_np, heatmap, alpha=0.45)
    
    # Kolom 1: Citra Asli
    axes[idx, 0].imshow(orig_np)
    axes[idx, 0].set_title(f"True: {true_cls}\nPred: {pred_cls} ({confidence:.1f}%)", fontsize=10)
    axes[idx, 0].axis("off")
    
    # Kolom 2: Heatmap
    axes[idx, 1].imshow(heatmap, cmap="jet")
    axes[idx, 1].set_title("Grad-CAM Heatmap", fontsize=10)
    axes[idx, 1].axis("off")
    
    # Kolom 3: Overlay Superimposed
    axes[idx, 2].imshow(overlay)
    axes[idx, 2].set_title("Superimposed Visualization", fontsize=10)
    axes[idx, 2].axis("off")

plt.suptitle(f"Visualisasi Grad-CAM pada Model Terbaik ({best_model_name})", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/gradcam_samples.png", dpi=300)
plt.show()


---
## 8. Analisis & Kesimpulan

### 1. Perbandingan Kinerja Model
- **Akurasi & Macro F1-Score:**
  Model **MobileNetV2 (Transfer Learning)** secara konsisten mengungguli **Custom CNN from Scratch** dengan margin akurasi dan Macro F1 yang signifikan. Hal ini membuktikan efektivitas *knowledge transfer* dari jutaan citra ImageNet ke domain klasifikasi pemandangan 6 kelas.
- **Efisiensi Parameter & Waktu Pelatihan:**
  Meskipun MobileNetV2 memiliki total parameter lebih besar (~2.26 juta vs ~450 ribu pada Custom CNN), jumlah *trainable parameter* pada tahap pertama sangat ringkas (~7.7 ribu), sehingga waktu konvergensi per epoch jauh lebih cepat dan stabil.

### 2. Analisis Kebingungan Antar-Kelas (Confusion Matrix)
- Pasangan kelas yang paling rentan mengalami salah prediksi (*misclassification*) adalah **glacier** dan **mountain**. Secara visual, kedua kelas tersebut memiliki kemiripan fitur dominan berupa lanskap bebatuan bersalju, lereng curam, dan pencahayaan langit putih-abu.
- Visualisasi Grad-CAM mengonfirmasi bahwa MobileNetV2 memfokuskan atensi pada tekstur spesifik: es gletser yang memantulkan cahaya vs siluet punggung gunung bersalju.

### 3. Keterbatasan Eksperimen (*Limitations*)
1. Ukuran input $150 \times 150$ piksel mempercepat pelatihan di Colab T4, tetapi mengorbankan sebagian detail frekuensi tinggi pada tekstur bangunan (*buildings*) dan jalan (*street*).
2. Data augmentasi yang diterapkan dibatasi pada rotasi, pergeseran horizontal, dan zoom ringan. Teknik augmentasi yang lebih agresif seperti CutMix atau MixUp belum dieksplorasi.

### 4. Rekomendasi Pengembangan Lanjut (*Future Work*)
1. Eksplorasi arsitektur backbone modern yang lebih kuat dan efisien seperti **EfficientNetV2-B0** atau **ConvNeXt-Tiny**.
2. Penerapan *Test-Time Augmentation* (TTA) untuk mendongkrak performa inferensi pada citra batas (*boundary samples*).
3. Hyperparameter search otomatis (menggunakan Keras Tuner) untuk optimasi learning rate schedule dan dropout rate secara sistematis.


In [ ]:
# Verifikasi Seluruh Artefak yang Dihasilkan
print("Daftar Berkas Hasil Eksperimen di /content/outputs:")
print("-" * 60)
for p in sorted(Path(OUTPUT_DIR).glob("*.*")):
    size_kb = p.stat().st_size / 1024
    print(f"  - {p.name:<38} ({size_kb:8.1f} KB)")
print("-" * 60)

# Opsional: Salin seluruh folder outputs ke Google Drive Anda
# Hapus tanda komentar (#) di bawah ini jika ingin mencadangkan ke Google Drive
"""
from google.colab import drive
drive.mount('/content/drive')
drive_backup_dir = "/content/drive/MyDrive/DL_Assignment_Outputs"
shutil.copytree(OUTPUT_DIR, drive_backup_dir, dirs_exist_ok=True)
print(f"[+] Seluruh artefak berhasil dicadangkan ke Google Drive: {drive_backup_dir}")
"""
